# Cloud interpretability run — Gemma-2-2b + Gemma Scope SAEs

**Mode 3** of `edtech-grading-transparency`. Run this on a cloud GPU (Colab / Kaggle / rented).

It loads Gemma-2-2b and Gemma Scope SAEs, runs mechanistic-interpretability analysis on the
example answers, and **exports artifacts** to `data/interp_artifacts/` so the offline **Mode 1**
proof on any laptop can replay genuine results.

See `docs/running-modes.md` for the full explanation.

> This notebook is intentionally minimal: the setup, auth, and export scaffolding are complete;
> the mech-interp analysis cell is marked `TODO` and is filled in as the interpretability modules
> land in `interpretability/`.

## 1. Install dependencies
Assumes the repo is available (e.g. `git clone` at the top of the notebook on Colab/Kaggle).

In [ ]:
# !git clone https://github.com/velezdeguevara/edtech-grading-transparency.git
# %cd edtech-grading-transparency
!pip install -q -r requirements.txt
!pip install -q -r requirements-interp.txt

## 2. Authenticate to Hugging Face (Gemma is gated)
Accept the Gemma-2-2b license on its model page first, then provide a token.
**Never commit your token.** On Colab, prefer the Secrets panel or `huggingface-cli login`.

In [ ]:
import os
from huggingface_hub import login

# Reads HF_TOKEN from the environment / Colab secrets; does not hard-code any secret.
token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)
else:
    print("Set HF_TOKEN (env var or Colab secret) before running the Gemma cells.")

## 3. Load Gemma-2-2b + Gemma Scope SAEs

In [ ]:
import torch
from transformer_lens import HookedTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = HookedTransformer.from_pretrained("gemma-2-2b", device=device)
# Gemma Scope SAEs are loaded via sae_lens where needed, e.g.:
# from sae_lens import SAE
# sae, cfg, sparsity = SAE.from_pretrained(release="gemma-scope-2b-pt-res", sae_id="...")

## 4. Run mech-interp on the example answers (TODO)
This cell will call the shared interpretability modules (`interpretability/`) — logit lens,
activation patching, attention analysis, SAE feature maps — against the fixtures in
`data/fixtures/`, using the same analysis code as Modes 1 and 2.

In [ ]:
# TODO: import and run the shared interpretability analysis against the fixtures.
# The analysis + escalation logic is defined once against the InterpBackend interface
# so this cell only supplies the real (torch) backend and collects results.
raise NotImplementedError("Interpretability analysis is wired in as the interpretability/ modules land.")

## 5. Export artifacts for Mode 1
Writes small artifact files to `data/interp_artifacts/` that the offline `ArtifactBackend` replays.
These replace the synthetic placeholders with genuine Gemma-derived captures.

In [ ]:
from pathlib import Path

artifacts_dir = Path("data/interp_artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
# TODO: serialise captured activations / attention / SAE features here, e.g. .npz + .json,
# in the schema the ArtifactBackend expects.
print("Export target:", artifacts_dir.resolve())